In [1]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow import keras
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

photos =  []
images = []

I0000 00:00:1776958778.890375   14284 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776958782.384254   14284 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776958790.781718   14284 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
#folders = listdir('img_align_celeba/img_align_celeba')

#len(folders)
for file in listdir('img_align_celeba/img_align_celeba' )[:20000]:
    photo = load_img('img_align_celeba/img_align_celeba/'+file, target_size=(128,128), color_mode='grayscale')
    photo = img_to_array(photo)
    images.append(file)
    photos.append(photo)


In [3]:
import pandas as pd

df_famosos = pd.read_csv('list_attr_celeba.csv')
df_famosos.head()

,image_id,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,...,Sideburns,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young
0,000001.jpg,-1,1,1,-1,-1,-1,-1,-1,-1,...,-1,1,1,-1,1,-1,1,-1,-1,1
1,000002.jpg,-1,-1,-1,1,-1,-1,-1,1,-1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
2,000003.jpg,-1,-1,-1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,1
3,000004.jpg,-1,-1,1,-1,-1,-1,-1,-1,-1,...,-1,-1,1,-1,1,-1,1,1,-1,1
4,000005.jpg,-1,1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1


In [4]:
len(images)

20000

In [5]:
df_famosos.shape

(202599, 41)

In [6]:
lista_atractivos = []
lista_calvos = []
lista_bolsas = []

In [7]:
for image in images:
    lista_atractivos.append(int(df_famosos[df_famosos['image_id'] == image]['Attractive'].to_string().split()[1]))
    lista_calvos.append(int(df_famosos[df_famosos['image_id'] == image]['Bald'].to_string().split()[1]))
    lista_bolsas.append(int(df_famosos[df_famosos['image_id'] == image]['Bags_Under_Eyes'].to_string().split()[1]))

In [8]:
df_famosos_filtrado = pd.DataFrame({'Attractive': lista_atractivos, 'Bald': lista_calvos, 'Bags_Under_Eyes': lista_bolsas})
df_famosos_filtrado.head()

,Attractive,Bald,Bags_Under_Eyes
0,1,-1,-1
1,1,-1,-1
2,1,-1,1
3,1,-1,-1
4,-1,-1,1


In [9]:
df_famosos_filtrado.replace(-1, 0, inplace=True)

In [10]:
photos = asarray(photos, dtype='float32') / 255.0
# photos = photos.reshape(len(photos), -1) / 255.0

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(photos, df_famosos_filtrado, test_size=0.2, random_state=42)

In [12]:
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(16, (3,3),activation='relu',input_shape=(X_train.shape[1:])))
model.add(keras.layers.MaxPool2D((2,2)))
model.add(keras.layers.Conv2D(32, (3,3),activation='relu'))
model.add(keras.layers.MaxPool2D((2,2)))
# model.add(keras.layers.Conv2D(64, (3,3),activation='relu'))
# model.add(keras.layers.MaxPool2D((2,2)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(100, activation='relu',kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(40, activation='relu', kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(3, activation='sigmoid', kernel_initializer='glorot_normal'))


/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1776959341.235714   14284 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [13]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.00001, beta_1=0.9, beta_2=0.999), loss='binary_crossentropy', metrics=['accuracy'])

In [14]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=5, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - accuracy: 0.8435 - loss: 0.4760 - val_accuracy: 0.8444 - val_loss: 0.4432
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.8390 - loss: 0.4193 - val_accuracy: 0.7962 - val_loss: 0.4231
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.8172 - loss: 0.4026 - val_accuracy: 0.8150 - val_loss: 0.4102
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.8006 - loss: 0.3887 - val_accuracy: 0.7688 - val_loss: 0.3936
Epoch 5/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.7913 - loss: 0.3773 - val_accuracy: 0.7856 - val_loss: 0.3806


In [15]:
import numpy as np
# y_test = np.array(y_test, dtype='int32')
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss: {loss}, Accuracy: {accuracy}')

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8083 - loss: 0.3721
Loss: 0.37210142612457275, Accuracy: 0.8082500100135803


In [16]:
y_pred = model.predict(X_test)
#pintar todas las predicciones
for i in range(len(y_pred)):
    print(f'Predicción: {np.round(y_pred[i])}, Real: {y_test.iloc[i].values}')

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Predicción: [0. 0. 0.], Real: [1 0 1]
Predicción: [0. 0. 0.], Real: [0 0 0]
Predicción: [0. 0. 0.], Real: [0 0 0]
Predicción: [0. 0. 0.], Real: [0 0 0]
Predicción: [0. 0. 0.], Real: [1 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicción: [1. 0. 0.], Real: [0 0 1]
Predicción: [0. 0. 0.], Real: [0 1 1]
Predicción: [0. 0. 0.], Real: [0 0 1]
Predicción: [1. 0. 0.], Real: [0 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicción: [0. 0. 0.], Real: [1 0 1]
Predicción: [1. 0. 0.], Real: [1 0 1]
Predicción: [0. 0. 0.], Real: [1 0 0]
Predicción: [0. 0. 0.], Real: [1 0 0]
Predicción: [0. 0. 0.], Real: [1 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicción: [0. 0. 0.], Real: [1 0 1]
Predicción: [1. 0. 0.], Real: [0 0 0]
Predicción: [0. 0. 0.], Real: [0 0 0]
Predicción: [0. 0. 0.], Real: [1 0 1]
Predicción: [0. 0. 0.], Real: [0 0 0]
Predicción: [1. 0. 0.], Real: [1 0 0]
Predicció